In [1]:
import sys
print(sys.executable)


c:\Users\juanc\Documents\Ausbildung_Informatik\1_Praktikum\Praktikum_Daten-_und_Prozessanalyse\Python_JNotebook\Qiskit\qiskit_env\Scripts\python.exe


In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
import timeit
from qiskit import QuantumCircuit, transpile
from qiskit_ibm_runtime import QiskitRuntimeService, Sampler

class IBMQuantumGrover:
    def __init__(self, api_token=None, instance=None, channel="ibm_quantum_platform"):
        # Service-Initialisierung mit API-Token, Kanal und Instanz-CRN
        self.service = QiskitRuntimeService(
            channel=channel,
            token=api_token,
            instance=instance
        )
        self.job_ids = {}

    def linear_search(self, items, target):
        """Klassische lineare Suche."""
        return items.index(target) if target in items else -1

    def create_grover_oracle(self, n_qubits, target_state):
        """Erstellt Orakel für Zielzustand |11...1>."""
        if n_qubits < 2:
            raise ValueError("Grover benötigt mindestens 2 Qubits!")
        qc = QuantumCircuit(n_qubits, name="oracle")
        if target_state == (2**n_qubits - 1):
            qc.x(range(n_qubits))
            qc.h(n_qubits-1)
            qc.mcx(list(range(n_qubits-1)), n_qubits-1)
            qc.h(n_qubits-1)
            qc.x(range(n_qubits))
        else:
            raise NotImplementedError("Orakel nur für |11...1> implementiert")
        return qc

    def run_grover(self, n_qubits, backend_name, shots=256):
        target_state = 2 ** n_qubits - 1
        qc = QuantumCircuit(n_qubits, n_qubits)
        qc.h(range(n_qubits))
        iterations = int(np.floor(np.pi / 4 * np.sqrt(2 ** n_qubits)))
        for _ in range(iterations):
            qc.append(self.create_grover_oracle(n_qubits, target_state), range(n_qubits))
            qc.h(range(n_qubits))
            qc.x(range(n_qubits))
            qc.h(n_qubits-1)
            qc.mcx(list(range(n_qubits-1)), n_qubits-1)
            qc.h(n_qubits-1)
            qc.x(range(n_qubits))
            qc.h(range(n_qubits))
        qc.measure(range(n_qubits), range(n_qubits))

        backend = self.service.backend(backend_name)
        # WICHTIG: Transpiliere die Schaltung explizit für das konkrete Backend!
        transpiled_qc = transpile(qc, backend=backend, optimization_level=3)
        start_time = time.time()

        sampler = Sampler(mode=backend)
        # Circuit als Liste übergeben!
        result = sampler.run([transpiled_qc], shots=shots).result()
        counts = result.quasi_distr[0]
        job_id = result.metadata.get("job_id", None)

        self.job_ids[job_id] = {
            "n_qubits": n_qubits,
            "backend": backend_name,
            "timestamp": start_time
        }
        return job_id, time.time() - start_time, transpiled_qc, counts, backend

    def compare_algorithms(self, max_qubits=5, backend_name=None):
        results = []
        for n in range(2, max_qubits + 1):
            size = 2 ** n
            print(f"\nVerarbeite Größe {size} ...")
            items = list(range(size))
            target = size - 1

            classical_time = np.mean([
                timeit.timeit(lambda: self.linear_search(items, target), number=100)
                for _ in range(3)
            ])
            # # # # # # Diese Zeile unterhalb einmal auskommentieren um den Job nicht nach IB zu senden bzw zu starten das Ganze# # # # # #
            job_id, quantum_time, _, quantum_counts, backend = self.run_grover(n, backend_name)
            results.append({
                "size": size,
                "classical_time": classical_time,
                "quantum_time": quantum_time,
                "job_id": job_id,
                "n_qubits": n,
                "counts": quantum_counts,
                "backend": backend.name
            })
            print(f"Klassisch: {classical_time:.4f}s | Quanten (Job {job_id}, Backend {backend.name}): {quantum_time:.4f}s")

        self.save_results(results)
        self.plot_results(results)
        return results

    def save_results(self, results, filename="quantum_results.json"):
        import json
        with open(filename, "w") as f:
            json.dump(results, f, indent=2)
        print(f"Ergebnisse gespeichert in {filename}")

    def plot_results(self, results):
        sizes = [r['size'] for r in results]
        classical = [r['classical_time'] for r in results]
        quantum = [r['quantum_time'] for r in results]
        plt.figure(figsize=(10, 6))
        plt.plot(sizes, classical, 'b-o', label='Lineare Suche (O(n))')
        plt.plot(sizes, quantum, 'r--o', label='Grover auf IBMQ (Job-Zeit)')
        plt.xscale('log', base=2)
        plt.yscale('log')
        plt.xlabel('Anzahl der Elemente (log₂-Skala)')
        plt.ylabel('Zeit (Sekunden, log-Skala)')
        plt.title('Leistungsvergleich: Klassisch vs. Quanten (IBM Hardware)')
        plt.legend()
        plt.grid(True, which="both", ls="--")
        plt.savefig('ibmq_grover_results.png', dpi=600, bbox_inches='tight')
        plt.show()

if __name__ == "__main__":
    API_TOKEN = ""
    INSTANCE = ""
    BACKEND_NAME = "ibm_torino"

    grover = IBMQuantumGrover(api_token=API_TOKEN, instance=INSTANCE)
    results = grover.compare_algorithms(max_qubits=5, backend_name=BACKEND_NAME)


In [16]:
import time
import numpy as np
import matplotlib.pyplot as plt
import timeit
from qiskit import QuantumCircuit, transpile
from qiskit_ibm_runtime import QiskitRuntimeService, Sampler
import json


class IBMQuantumGrover:
    def __init__(self, api_token=None, instance=None, channel="ibm_quantum_platform"):
        self.service = QiskitRuntimeService(
            channel=channel,
            token=api_token,
            instance=instance
        )
        self.job_ids = {}

    def linear_search(self, items, target):
        return items.index(target) if target in items else -1

    def create_grover_oracle(self, n_qubits, target_state):
        qc = QuantumCircuit(n_qubits, name="oracle")
        if n_qubits < 2:
            raise ValueError("Grover benötigt mindestens 2 Qubits!")
        if target_state == (2**n_qubits - 1):
            qc.x(range(n_qubits))
            qc.h(n_qubits-1)
            if n_qubits > 2:
                qc.mcx(list(range(n_qubits-1)), n_qubits-1)
            else:
                qc.cx(0, 1)
            qc.h(n_qubits-1)
            qc.x(range(n_qubits))
        else:
            raise NotImplementedError("Orakel nur für |11...1> implementiert")
        return qc

    def run_grover(self, n_qubits, backend_name, shots=256):
        target_state = 2 ** n_qubits - 1
        qc = QuantumCircuit(n_qubits, n_qubits)
        qc.h(range(n_qubits))
        iterations = int(np.floor(np.pi / 4 * np.sqrt(2 ** n_qubits)))

        for _ in range(iterations):
            qc.append(self.create_grover_oracle(n_qubits, target_state), range(n_qubits))
            qc.h(range(n_qubits))
            qc.x(range(n_qubits))
            qc.h(n_qubits-1)
            if n_qubits > 2:
                qc.mcx(list(range(n_qubits-1)), n_qubits-1)
            else:
                qc.cx(0, 1)
            qc.h(n_qubits-1)
            qc.x(range(n_qubits))
            qc.h(range(n_qubits))

        qc.measure(range(n_qubits), range(n_qubits))

        backend = self.service.backend(backend_name)
        transpiled_qc = transpile(qc, backend=backend, optimization_level=3)
        start_time = time.time()

        sampler = Sampler(mode=backend)
        result = sampler.run([transpiled_qc], shots=shots).result()
        counts = result.quasi_dists[0]  # Je nach Version ggf. anpassen
        job_id = result.metadata.get("job_id", None)

        self.job_ids[job_id] = {
            "n_qubits": n_qubits,
            "backend": backend_name,
            "timestamp": start_time
        }
        return job_id, time.time() - start_time, transpiled_qc, counts, backend

    def compare_algorithms(self, max_qubits=5, backend_name=None):
        results = []
        for n in range(2, max_qubits + 1):
            size = 2 ** n
            print(f"\nVerarbeite Größe {size} ...")
            items = list(range(size))
            target = size - 1

            classical_time = np.mean([
                timeit.timeit(lambda: self.linear_search(items, target), number=100)
                for _ in range(3)
            ])

            job_id, quantum_time, _, quantum_counts, backend = self.run_grover(n, backend_name)
            results.append({
                "size": size,
                "classical_time": classical_time,
                "quantum_time": quantum_time,
                "job_id": job_id,
                "n_qubits": n,
                "counts": quantum_counts,
                "backend": backend.name
            })
            print(f"Klassisch: {classical_time:.4f}s | Quanten (Job {job_id}, Backend {backend.name}): {quantum_time:.4f}s")

        self.save_results(results)
        return results

    def save_results(self, results, filename="quantum_results.json"):
        with open(filename, "w") as f:
            json.dump(results, f, indent=2)
        print(f"Ergebnisse gespeichert in {filename}")


In [12]:
if __name__ == "__main__":
    API_TOKEN = "hc_jCRxZWVqz8KZuZY58kmtbiC81I1tuGuKrU4a-VOGF"
    INSTANCE = "crn:v1:bluemix:public:quantum-computing:us-east:a/71e0d8f4996f4919a7a1f5a17593eac9:817179d7-c733-47f0-89fa-64f2696e053c::"
    BACKEND_NAME = "ibm_torino"  # Beispiel

    grover = IBMQuantumGrover(api_token=API_TOKEN, instance=INSTANCE)
    grover.compare_algorithms(max_qubits=5, backend_name=BACKEND_NAME)



Verarbeite Größe 4 ...


AttributeError: 'PrimitiveResult' object has no attribute 'quasi_dists'

In [13]:
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService(
    channel='ibm_quantum_platform',
    instance='crn:v1:bluemix:public:quantum-computing:us-east:a/71e0d8f4996f4919a7a1f5a17593eac9:817179d7-c733-47f0-89fa-64f2696e053c::'
)
job = service.job('d2eoa5ms6qcs738edlq0')
job_result = job.result()

# To get counts for a particular pub result, use
#
# pub_result = job_result[<idx>].data.<classical register>.get_counts()
#
# where <idx> is the index of the pub and <classical register> is the name of the classical register.
# You can use circuit.cregs to find the name of the classical registers.

In [17]:
import json
import matplotlib.pyplot as plt
import numpy as np


class IBMQuantumGroverAnalysis:
    def __init__(self, results_file="quantum_results.json"):
        with open(results_file, "r") as f:
            self.results = json.load(f)

    def plot_results(self):
        sizes = [r['size'] for r in self.results]
        classical = [r['classical_time'] for r in self.results]
        quantum = [r['quantum_time'] for r in self.results]

        plt.figure(figsize=(10, 6))
        plt.plot(sizes, classical, 'b-o', label='Lineare Suche (O(n))')
        plt.plot(sizes, quantum, 'r--o', label='Grover auf IBMQ (Job-Zeit)')
        plt.xscale('log', base=2)
        plt.yscale('log')
        plt.xlabel('Anzahl der Elemente (log₂-Skala)')
        plt.ylabel('Zeit (Sekunden, log-Skala)')
        plt.title('Leistungsvergleich: Klassisch vs. Quanten (IBM Hardware)')
        plt.legend()
        plt.grid(True, which="both", ls="--")
        plt.savefig('ibmq_grover_results.png', dpi=600, bbox_inches='tight')
        plt.show()


if __name__ == "__main__":
    analysis = IBMQuantumGroverAnalysis(results_file="quantum_results.json")
    analysis.plot_results()


FileNotFoundError: [Errno 2] No such file or directory: 'quantum_results.json'

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
import timeit
from qiskit import QuantumCircuit, transpile
from qiskit_ibm_runtime import QiskitRuntimeService, Sampler
import json


class IBMQuantumGrover:
    def __init__(self, api_token=None, instance=None, channel="ibm_quantum_platform"):
        self.service = QiskitRuntimeService(
            channel=channel,
            token=api_token,
            instance=instance
        )
        self.job_ids = {}

    def linear_search(self, items, target):
        return items.index(target) if target in items else -1

    def create_grover_oracle(self, n_qubits, target_state):
        qc = QuantumCircuit(n_qubits, name="oracle")
        if n_qubits < 2:
            raise ValueError("Grover benötigt mindestens 2 Qubits!")
        if target_state == (2**n_qubits - 1):
            qc.x(range(n_qubits))
            qc.h(n_qubits-1)
            if n_qubits > 2:
                qc.mcx(list(range(n_qubits-1)), n_qubits-1)
            else:
                qc.cx(0, 1)
            qc.h(n_qubits-1)
            qc.x(range(n_qubits))
        else:
            raise NotImplementedError("Orakel nur für |11...1> implementiert")
        return qc

    def run_grover(self, n_qubits, backend_name, shots=256):
        target_state = 2 ** n_qubits - 1
        qc = QuantumCircuit(n_qubits, n_qubits)
        qc.h(range(n_qubits))
        iterations = int(np.floor(np.pi / 4 * np.sqrt(2 ** n_qubits)))

        for _ in range(iterations):
            qc.append(self.create_grover_oracle(n_qubits, target_state), range(n_qubits))
            qc.h(range(n_qubits))
            qc.x(range(n_qubits))
            qc.h(n_qubits-1)
            if n_qubits > 2:
                qc.mcx(list(range(n_qubits-1)), n_qubits-1)
            else:
                qc.cx(0, 1)
            qc.h(n_qubits-1)
            qc.x(range(n_qubits))
            qc.h(range(n_qubits))

        qc.measure(range(n_qubits), range(n_qubits))

        backend = self.service.backend(backend_name)
        transpiled_qc = transpile(qc, backend=backend, optimization_level=3)
        start_time = time.time()

        sampler = Sampler(mode=backend)
        result = sampler.run([transpiled_qc], shots=shots).result()
        counts = result.quasi_dists[0]  # Je nach Version ggf. anpassen
        job_id = result.metadata.get("job_id", None)

        self.job_ids[job_id] = {
            "n_qubits": n_qubits,
            "backend": backend_name,
            "timestamp": start_time
        }
        return job_id, time.time() - start_time, transpiled_qc, counts, backend

    def compare_algorithms(self, max_qubits=5, backend_name=None):
        results = []
        for n in range(2, max_qubits + 1):
            size = 2 ** n
            print(f"\nVerarbeite Größe {size} ...")
            items = list(range(size))
            target = size - 1

            classical_time = np.mean([
                timeit.timeit(lambda: self.linear_search(items, target), number=100)
                for _ in range(3)
            ])

            job_id, quantum_time, _, quantum_counts, backend = self.run_grover(n, backend_name)
            results.append({
                "size": size,
                "classical_time": classical_time,
                "quantum_time": quantum_time,
                "job_id": job_id,
                "n_qubits": n,
                "counts": quantum_counts,
                "backend": backend.name
            })
            print(f"Klassisch: {classical_time:.4f}s | Quanten (Job {job_id}, Backend {backend.name}): {quantum_time:.4f}s")

        self.save_results(results)
        return results

    def save_results(self, results, filename="quantum_results.json"):
        with open(filename, "w") as f:
            json.dump(results, f, indent=2)
        print(f"Ergebnisse gespeichert in {filename}")

if __name__ == "__main__":
    API_TOKEN = ""
    INSTANCE = ""
    BACKEND_NAME = "ibm_torino"  # Beispiel

    grover = IBMQuantumGrover(api_token=API_TOKEN, instance=INSTANCE)
    grover.compare_algorithms(max_qubits=5, backend_name=BACKEND_NAME)


import json
import matplotlib.pyplot as plt
import numpy as np


class IBMQuantumGroverAnalysis:
    def __init__(self, results_file="quantum_results.json"):
        with open(results_file, "r") as f:
            self.results = json.load(f)

    def plot_results(self):
        sizes = [r['size'] for r in self.results]
        classical = [r['classical_time'] for r in self.results]
        quantum = [r['quantum_time'] for r in self.results]

        plt.figure(figsize=(10, 6))
        plt.plot(sizes, classical, 'b-o', label='Lineare Suche (O(n))')
        plt.plot(sizes, quantum, 'r--o', label='Grover auf IBMQ (Job-Zeit)')
        plt.xscale('log', base=2)
        plt.yscale('log')
        plt.xlabel('Anzahl der Elemente (log₂-Skala)')
        plt.ylabel('Zeit (Sekunden, log-Skala)')
        plt.title('Leistungsvergleich: Klassisch vs. Quanten (IBM Hardware)')
        plt.legend()
        plt.grid(True, which="both", ls="--")
        plt.savefig('ibmq_grover_results.png', dpi=600, bbox_inches='tight')
        plt.show()


if __name__ == "__main__":
    analysis = IBMQuantumGroverAnalysis(results_file="quantum_results.json")
    analysis.plot_results()



In [2]:
# Daten/Ergebnisse aus der Cloud laden
# 

from qiskit_ibm_runtime import QiskitRuntimeService

# Deine Zugangsdaten
API_TOKEN = "hc_jCRxZWVqz8KZuZY58kmtbiC81I1tuGuKrU4a-VOGF"
INSTANCE = "crn:v1:bluemix:public:quantum-computing:us-east:a/71e0d8f4996f4919a7a1f5a17593eac9:817179d7-c733-47f0-89fa-64f2696e053c::"
CHANNEL = "ibm_quantum_platform"

# Job-ID der erfolgten Berechnung
job_id = "d2e8qknl2k0s73ai4phg"

# Qiskit-IBM Service initialisieren
service = QiskitRuntimeService(
    channel=CHANNEL,
    token=API_TOKEN,
    instance=INSTANCE
)

# Job aus der Cloud laden
job = service.job(job_id)
result = job.result()

# Ergebnisse anzeigen/verarbeiten
print(result)
# Beispielsweise die Häufigkeitsverteilung der gemessenen Zustände:
if hasattr(result, "quasi_dists"):
    print(result.quasi_dists[0])
elif hasattr(result, "get_counts"):
    print(result.get_counts())


PrimitiveResult([SamplerPubResult(data=DataBin(c=BitArray(<shape=(), num_shots=256, num_bits=2>)), metadata={'circuit_metadata': {}})], metadata={'execution': {'execution_spans': ExecutionSpans([DoubleSliceSpan(<start='2025-08-13 13:04:22', stop='2025-08-13 13:04:25', size=256>)])}, 'version': 2})


In [4]:
import json

# Angenommen, result = job.result()

try:
    # Versuche, die wahrscheinlichkeitsverteilung zu extrahieren (Grover/Sampler)
    data_to_save = result.quasi_dists[0]
except AttributeError:
    try:
        # Versuche, Messwertzählungen zu extrahieren
        data_to_save = result.get_counts()
    except AttributeError:
        try:
            # Alternatives Attribut für rohe Daten
            data_to_save = result.data()
        except AttributeError:
            # Als letztes: das Ergebnisobjekt serialisieren
            data_to_save = str(result)

with open("ibmq_job_result.json", "w") as f:
    json.dump(data_to_save, f, indent=2)

print("Ergebnis in ibmq_job_result.json gespeichert.")


Ergebnis in ibmq_job_result.json gespeichert.


In [1]:
import numpy as np
import json

# Annahme: sampler_pub_result ist das result
bit_array = sampler_pub_result.data.c  # 'c' für klassisches Register

# Nutze die empfohlene Methode:
bitstrings = bit_array.get_bitstrings()  # Gibt eine Liste von Bitstrings zurück, z.B. ['11', '10', ...]

# Häufigkeiten zählen:
unique, counts = np.unique(bitstrings, return_counts=True)
# Konvertiere numpy int64 zu normalem int
count_dict = {str(k): int(v) for k, v in zip(unique, counts)}

with open("ibmq_counts_result.json", "w") as f:
    json.dump(count_dict, f, indent=2)

print(count_dict)

NameError: name 'sampler_pub_result' is not defined